In [1]:
import numpy as np
import pandas as pd
import os
import openpyxl
import re

In [2]:
data_path = os.path.join(os.getcwd(), "..", "data", "training")
raw_data_path = os.path.join(data_path, "raw")
converted_data_path = os.path.join(data_path, "converted")
os.makedirs(converted_data_path, exist_ok=True)

In [3]:
raw_files = os.listdir(raw_data_path)
raw_files = [file for file in raw_files if file.endswith(".xlsx")]
raw_files

['raw2.xlsx', 'raw.xlsx']

In [4]:
file_mapping = dict()
for file in raw_files:
    if re.match(r".*\d+.*", file.lower(), re.IGNORECASE):
        name = f"series_{re.search(r".*(\d+).*", file.lower(), flags=re.IGNORECASE).group(1)}"
    else:
        name = "series_1"
    file_mapping[file] = name
    print(f"Mapped {file} to {name}.")

file_mapping

Mapped raw2.xlsx to series_2.
Mapped raw.xlsx to series_1.


{'raw2.xlsx': 'series_2', 'raw.xlsx': 'series_1'}

In [5]:
excel_files: list[tuple[str, pd.ExcelFile | None]] = [("", None) for _ in range(len(raw_files))]
for i, file in enumerate(raw_files):
    print(f"Loading {file}...")
    excel_files[i] = (file, pd.ExcelFile(os.path.join(raw_data_path, file), engine="openpyxl", engine_kwargs={"read_only": True}))
    print(f"Loaded {file} successfully.")

Loading raw2.xlsx...


Loaded raw2.xlsx successfully.
Loading raw.xlsx...
Loaded raw.xlsx successfully.


In [11]:
sheet_lists: list[tuple[str, list[str]]] = [(wb[0], [str(sheet) for sheet in wb[1].sheet_names]) for wb in excel_files if wb[1] is not None]
sheet_lists

[('raw2.xlsx', ['Alle', 'Wurst1', 'Wurst2', 'Wurst3', 'Wurst4', 'Wurst5']),
 ('raw.xlsx',
  ['Filtered',
   '+ Annotation',
   'Messwerte Wurst 1, 1. Messreihe',
   'Messwerte Wurst 2, 1. Messreihe',
   'Messwerte Wurst 3, 1. Messreihe',
   'Messwerte Wurst 4, 1. Messreihe',
   'Messwerte Wurst 5, 1. Messreihe'])]

In [14]:
target_sheets = []
for (file, sheet_list) in sheet_lists:
    for sheet in sheet_list:
        if re.match(r".*alle.*", sheet.lower(), re.IGNORECASE) or re.match(r".*annotation.*", sheet.lower(), re.IGNORECASE):
            target_sheets.append(sheet)

target_sheets

['Alle', '+ Annotation']

In [16]:
sensor_mapping = {
    "WS202": r".*milesight.*",
    "LTC-FS": r".*wurst[^\d]*(\d+).*feuchtigkeit.*",
    "LSE01": r".*wurst[^\d]*(\d+).*temperatur.*"
}
sensor_mapping

{'WS202': '.*milesight.*',
 'LTC-FS': '.*wurst[^\\d]*(\\d+).*feuchtigkeit.*',
 'LSE01': '.*wurst[^\\d]*(\\d+).*temperatur.*'}

In [ ]:
for file, sheet_list in sheet_lists:
    for sheet in sheet_list:
        if sheet not in target_sheets:
            print(f"Skipping {sheet} in {file} as it is not a target sheet.")
            continue
        print(f"Processing {sheet} in {file}...")
        df = pd.read_excel(os.path.join(raw_data_path, file), sheet_name=sheet
                            , engine="openpyxl", engine_kwargs={"read_only": True})
        print(f"Loaded {sheet} in {file} successfully.")
        distinct_measurement_names = df["Gerät"].unique()
        print(f"Found distinct measurement names in {sheet} of {file}: {distinct_measurement_names}")
        target_measurements = dict()
        sensor_files = dict()
        for measurement_name in distinct_measurement_names:
            if not re.match(r".*wurst.*", measurement_name.lower(), re.IGNORECASE):
                
            for sensor_name, pattern in sensor_mapping.items():
                re_match = re.match(pattern, measurement_name.lower(), re.IGNORECASE)
                if re_match:
                    
                    target_measurements[measurement_name] = sensor_name
                    print(f"Mapped measurement name {measurement_name} to sensor {sensor_name} in {sheet} of {file}.")
                    break
            else:
                print(f"No mapping found for measurement name {measurement_name} in {sheet} of {file}.")

Skipping Alle in raw2.xlsx as it is not mapped.
Processing Wurst1 in raw2.xlsx and saving as series_2_sausage_1.csv...


Saved Wurst1 in raw2.xlsx as series_2_sausage_1.csv successfully.
Processing Wurst2 in raw2.xlsx and saving as series_2_sausage_2.csv...
Saved Wurst2 in raw2.xlsx as series_2_sausage_2.csv successfully.
Processing Wurst3 in raw2.xlsx and saving as series_2_sausage_3.csv...
Saved Wurst3 in raw2.xlsx as series_2_sausage_3.csv successfully.
Processing Wurst4 in raw2.xlsx and saving as series_2_sausage_4.csv...
Saved Wurst4 in raw2.xlsx as series_2_sausage_4.csv successfully.
Processing Wurst5 in raw2.xlsx and saving as series_2_sausage_5.csv...
Saved Wurst5 in raw2.xlsx as series_2_sausage_5.csv successfully.
Skipping Filtered in raw.xlsx as it is not mapped.
Skipping + Annotation in raw.xlsx as it is not mapped.
Processing Messwerte Wurst 1, 1. Messreihe in raw.xlsx and saving as series_1_sausage_1.csv...
Saved Messwerte Wurst 1, 1. Messreihe in raw.xlsx as series_1_sausage_1.csv successfully.
Processing Messwerte Wurst 2, 1. Messreihe in raw.xlsx and saving as series_1_sausage_2.csv...


In [ ]:
for workbook in excel_files:
    workbook[1].close()